# CutMix — Regularization Strategy to Train Strong Classifiers with Localizable Features (2019)

_Papers · Computer Vision_

**Cut a rectangular patch out of one training image, paste in the same patch from a second image, and mix their labels by the AREA each image now covers.**

---

This notebook is a practice scaffold for the **CutMix — Regularization Strategy to Train Strong Classifiers with Localizable Features (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import math, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)

# ---------- CutMix from scratch (Yun et al., 2019, Section 3.1 + Appendix A) ----------
def rand_bbox(W, H, lam):
    # Eq. (2): sides scale with sqrt(1-lam) so the patch area is the (1-lam) fraction.
    cut = math.sqrt(1.0 - lam)
    cw, ch = int(W * cut), int(H * cut)
    cx, cy = np.random.randint(W), np.random.randint(H)        # random center r_x, r_y
    x1 = int(np.clip(cx - cw // 2, 0, W)); x2 = int(np.clip(cx + cw // 2, 0, W))
    y1 = int(np.clip(cy - ch // 2, 0, H)); y2 = int(np.clip(cy + ch // 2, 0, H))
    return x1, y1, x2, y2

def cutmix_batch(x, y, alpha=1.0):
    N, C, H, W = x.shape
    lam = float(np.random.beta(alpha, alpha))                  # lam ~ Beta(a,a); a=1 -> uniform
    perm = torch.randperm(N)                                   # the "image B" partner for each image
    x = x.clone()
    x1, y1, x2, y2 = rand_bbox(W, H, lam)
    x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]           # Eq.(1): paste B's patch into A
    lam = 1.0 - (x2 - x1) * (y2 - y1) / (W * H)                # Appendix A: re-fit lam to EXACT area
    return x, y, y[perm], lam, (x1, y1, x2, y2)

# ---------- VERIFY the area identity numerically: re-fit lam == fraction of un-replaced A-pixels ----------
# Build the ACTUAL binary mask M (1 = keep A, 0 = pasted B), then COUNT the kept pixels independently
# of the closed-form re-fit. If the identity 1 - area/(W*H) == (kept pixels)/(W*H) holds, the two agree.
np.random.seed(7); max_err = 0.0
for _ in range(2000):
    lam0 = float(np.random.beta(1, 1))
    bx1, by1, bx2, by2 = rand_bbox(16, 16, lam0)
    M = np.ones((16, 16))                                      # mask: start all-A
    M[by1:by2, bx1:bx2] = 0.0                                  # zero out the pasted-B rectangle
    frac_A = M.mean()                                          # COUNT kept A-pixels, independently
    lam_refit = 1.0 - (bx2 - bx1) * (by2 - by1) / (16 * 16)    # Appendix-A closed form
    max_err = max(max_err, abs(lam_refit - frac_A))
print("max |refit_lambda - counted_A_pixel_fraction| over 2000 boxes:", max_err)   # expect exactly 0.0

# ---------- Contrast CutMix vs Mixup on the same image pair ----------
xa = torch.zeros(1, 3, 16, 16); xa[:, :, 2:8, 2:8] = 1.0       # image A: blob top-left
xb = torch.zeros(1, 3, 16, 16); xb[:, :, 8:14, 8:14] = 1.0     # image B: blob bottom-right
lam = 0.5
mixup_img = lam * xa + (1 - lam) * xb                          # faded double-exposure
cut = xa.clone(); cut[:, :, 8:16, 8:16] = xb[:, :, 8:16, 8:16] # real pixels, spliced
print("Mixup pixels are faded? unique vals:", torch.unique(mixup_img).tolist())  # has 0.5 -> faded
print("CutMix pixels stay real? unique vals:", torch.unique(cut).tolist())       # only 0.0 / 1.0

# ---------- Recompute the worked example: 32x32, lambda=0.6 ----------
W2 = H2 = 32; lam_s = 0.6
rw = int(round(W2 * math.sqrt(1 - lam_s))); rh = int(round(H2 * math.sqrt(1 - lam_s)))
area = rw * rh; lam_adj = 1 - area / (W2 * H2)
print(f"worked: rw=rh={rw}, area={area}/{W2*H2}, refit_lambda={lam_adj:.4f}, "
      f"label={lam_adj:.4f}*yA+{1-lam_adj:.4f}*yB")   # 20, 400/1024, 0.6094

# ---------- Train a tiny CNN: none vs mixup vs cutmix on an overfitting-prone toy task ----------
C, H, W = 3, 16, 16
def make(n, cls, seed):
    g = torch.Generator().manual_seed(seed)
    x = 1.1 * torch.randn(n, C, H, W, generator=g)             # strong noise -> easy to overfit
    if cls == 0: x[:, :, 2:8, 2:8]  += 0.6                      # faint localized blob, class 0
    else:        x[:, :, 8:14, 8:14] += 0.6                     # faint localized blob, class 1
    return x

Xtr = torch.cat([make(20, 0, 1), make(20, 1, 2)])              # only 40 training images
ytr = torch.cat([torch.zeros(20), torch.ones(20)]).long()
Xte = torch.cat([make(500, 0, 300), make(500, 1, 400)])        # 1000 fresh test images
yte = torch.cat([torch.zeros(500), torch.ones(500)]).long()

def net():
    return nn.Sequential(nn.Conv2d(3, 6, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(6, 12, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                         nn.Flatten(), nn.Linear(12, 2))

def train(mode, epochs=140, bs=15):
    torch.manual_seed(0); np.random.seed(0)                    # same init/data for all three
    m = net(); opt = torch.optim.Adam(m.parameters(), lr=0.012)
    for _ in range(epochs):
        perm = torch.randperm(Xtr.shape[0])
        for i in range(0, Xtr.shape[0], bs):
            idx = perm[i:i+bs]; xb_, yb = Xtr[idx], ytr[idx]; opt.zero_grad()
            if mode == "none":
                loss = F.cross_entropy(m(xb_), yb)
            elif mode == "mixup":
                lam = float(np.random.beta(1, 1)); r = torch.randperm(xb_.size(0))
                mix = lam * xb_ + (1 - lam) * xb_[r]; out = m(mix)
                loss = lam * F.cross_entropy(out, yb) + (1 - lam) * F.cross_entropy(out, yb[r])
            else:  # cutmix
                xc, ya, yb2, lam, _ = cutmix_batch(xb_, yb)
                out = m(xc)
                loss = lam * F.cross_entropy(out, ya) + (1 - lam) * F.cross_entropy(out, yb2)
            loss.backward(); opt.step()
    with torch.no_grad():
        tr = (m(Xtr).argmax(1) == ytr).float().mean().item()
        te = (m(Xte).argmax(1) == yte).float().mean().item()
    return tr, te

for mode in ["none", "mixup", "cutmix"]:
    tr, te = train(mode)
    print(f"{mode:7s} train_acc={tr:.3f}  test_acc={te:.3f}")
# expect (our run): none train 1.000/test ~0.53, mixup ~0.85, cutmix ~0.84 -> augmentation closes the gap

# ---------- ABLATION: skip the lambda re-fit -> how wrong is the label? ----------
np.random.seed(3); diffs = []
for _ in range(5000):
    lam0 = float(np.random.beta(1, 1))
    bx1, by1, bx2, by2 = rand_bbox(16, 16, lam0)
    true_B_frac = (bx2 - bx1) * (by2 - by1) / (16 * 16)        # what is REALLY pasted
    raw_B_frac  = 1 - lam0                                     # what the un-adjusted label assumes
    diffs.append(abs(true_B_frac - raw_B_frac))
print(f"un-adjusted label drift: mean={np.mean(diffs):.4f}, max={np.max(diffs):.4f}")

## Visualize the data & results

_Does CutMix keep pixels locally real (unlike Mixup's blend), and does training with it close the overfitting gap of a tiny CNN — and how wrong is the label if you skip the area-adjustment?_

In [ ]:
import math, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F

# ---- LEFT: none vs mixup vs cutmix test accuracy on an overfitting-prone toy task ----
C, H, W = 3, 16, 16
def make(n, cls, seed):
    g = torch.Generator().manual_seed(seed)
    x = 1.1 * torch.randn(n, C, H, W, generator=g)
    if cls == 0: x[:, :, 2:8, 2:8]  += 0.6
    else:        x[:, :, 8:14, 8:14] += 0.6
    return x
Xtr = torch.cat([make(20,0,1), make(20,1,2)]); ytr = torch.cat([torch.zeros(20), torch.ones(20)]).long()
Xte = torch.cat([make(500,0,300), make(500,1,400)]); yte = torch.cat([torch.zeros(500), torch.ones(500)]).long()

def rand_bbox(W, H, lam):
    cut = math.sqrt(1-lam); cw, ch = int(W*cut), int(H*cut)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1=int(np.clip(cx-cw//2,0,W)); x2=int(np.clip(cx+cw//2,0,W))
    y1=int(np.clip(cy-ch//2,0,H)); y2=int(np.clip(cy+ch//2,0,H))
    return x1,y1,x2,y2
def net():
    return nn.Sequential(nn.Conv2d(3,6,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(6,12,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                         nn.Flatten(), nn.Linear(12,2))
def train(mode, epochs=140, bs=15):
    torch.manual_seed(0); np.random.seed(0)
    m = net(); opt = torch.optim.Adam(m.parameters(), lr=0.012)
    for _ in range(epochs):
        perm = torch.randperm(Xtr.shape[0])
        for i in range(0, Xtr.shape[0], bs):
            idx = perm[i:i+bs]; xb = Xtr[idx].clone(); yb = ytr[idx]; opt.zero_grad()
            if mode == "none": loss = F.cross_entropy(m(xb), yb)
            elif mode == "mixup":
                lam = float(np.random.beta(1,1)); r = torch.randperm(xb.size(0))
                out = m(lam*xb + (1-lam)*xb[r])
                loss = lam*F.cross_entropy(out, yb) + (1-lam)*F.cross_entropy(out, yb[r])
            else:
                lam = float(np.random.beta(1,1)); r = torch.randperm(xb.size(0))
                x1,y1,x2,y2 = rand_bbox(W,H,lam); xb[:,:,y1:y2,x1:x2] = xb[r,:,y1:y2,x1:x2]
                lam = 1 - (x2-x1)*(y2-y1)/(W*H); out = m(xb)
                loss = lam*F.cross_entropy(out, yb) + (1-lam)*F.cross_entropy(out, yb[r])
            loss.backward(); opt.step()
    return (m(Xte).argmax(1) == yte).float().mean().item()
for mode in ["none","mixup","cutmix"]:
    print(mode, "test_acc =", round(train(mode), 3))   # ~0.533 / 0.846 / 0.835

# ---- MIDDLE: faded-pixel fraction, CutMix vs Mixup ----
xa = torch.zeros(1,3,16,16); xa[:,:,2:8,2:8] = 1.0
xb = torch.zeros(1,3,16,16); xb[:,:,8:14,8:14] = 1.0
mixup = 0.5*xa + 0.5*xb
cut = xa.clone(); cut[:,:,8:16,8:16] = xb[:,:,8:16,8:16]
faded = lambda im: ((im != 0) & (im != 1)).float().mean().item()   # pixels that are neither source value
print("faded fraction  CutMix:", faded(cut), " Mixup:", faded(mixup))  # 0.0  vs  1.0

# ---- RIGHT: label drift if you skip the lambda re-fit ----
np.random.seed(3); diffs = []
for _ in range(5000):
    lam = float(np.random.beta(1,1)); x1,y1,x2,y2 = rand_bbox(16,16,lam)
    diffs.append(abs((x2-x1)*(y2-y1)/256 - (1-lam)))
print("label drift mean:", round(float(np.mean(diffs)),4), "max:", round(float(np.max(diffs)),4))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** On a $24\times24$ image you sample $\lambda=0.75$. Compute the planned patch size $r_w,r_h$, the patch area, and (assuming the patch is fully inside) the re-fit $\lambda$ and the mixed label weights.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Patch sides: $r_w=r_h=24\sqrt{1-0.75}=24\sqrt{0.25}=24(0.5)=12$ px. — _$\sqrt{1-\lambda}$ scales each side so the area is the $1-\lambda$ fraction._
- Patch area $=12\times12=144$ px, out of $24\times24=576$. — _Area is width times height._
- Re-fit $\lambda=1-144/576=1-0.25=0.75$. — _No clipping and no rounding here, so it matches the sampled value exactly._

**Answer:** $r_w=r_h=12$ px, patch area $=144/576=25\%$, re-fit $\lambda=0.75$, so the label is $0.75\,y_A+0.25\,y_B$ — image A keeps three-quarters of the area and three-quarters of the label.

</details>

**Problem 2.** The patch is placed near a corner and clipped, so the actual replaced region is only $10\times8$ pixels on a $32\times32$ image, even though it was planned for $\lambda=0.6$. What label should you use, and what goes wrong if you keep $\lambda=0.6$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Actual swapped area $=10\times8=80$ px out of $1024$. — _Use the clipped corners, not the planned size._
- Re-fit $\lambda=1-80/1024=1-0.078=0.922$. — _Only ~8% of the image is now image B, so A should keep ~92% of the label._
- Compare to the un-adjusted $\lambda=0.6$. — _That would claim image B owns 40% of the label while it really owns ~8%._

**Answer:** Use $\tilde{y}=0.922\,y_A+0.078\,y_B$. Keeping $\lambda=0.6$ would tell the network the image is 40% class B when only 8% of its pixels are B — a badly mislabelled target. This is exactly why the paper re-fits $\lambda$ to the exact area ratio.

</details>

**Problem 3.** Ablation: in the CODE, run CutMix but SKIP the $\lambda$ area-adjustment (use the raw sampled $\lambda$ for the label). Measure how far the raw label weight drifts from the true pasted-area fraction across many random boxes. What does the ablation show, and why does the adjustment matter most near image borders?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For 5000 random boxes on a $16\times16$ image, sample $\lambda$, clip the box, and compare the TRUE B-area fraction $(x_2-x_1)(y_2-y_1)/(WH)$ against the raw assumption $1-\lambda$. — _Isolates the one quantity the adjustment fixes — the label's area weight._
- Average the absolute mismatch and record the maximum. — _Quantifies how wrong the un-adjusted label is on average and at worst._
- Note which boxes have the largest mismatch. — _Boxes whose random center is near an edge get clipped the most, so their true area shrinks well below $1-\lambda$._

**Answer:** In our run the un-adjusted label weight is off from the true pasted-area fraction by ~0.22 on average and up to ~0.78 of the whole image (see CODEVIZ — our small run, not the paper's number). The error is largest for boxes whose center falls near a border, because clipping removes part of the planned patch so far less of image B is actually present than $1-\lambda$ claims. The re-fit line $\lambda=1-(x_2-x_1)(y_2-y_1)/(WH)$ removes this mismatch exactly — the mechanism check confirms the re-fit $\lambda$ equals the true image-A pixel fraction with error 0.

</details>